# Unitree G1: GPU RL pipeline (MJX + custom PPO)

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, **Persistence = Files**.

Every session: run sections 1–2, then whichever phase cells you need. Cells marked ⏱ <5 min are interactive checks; cells marked 🌙 are long jobs to run with **Save Version → Save & Run All**.

## 1. Get / refresh the code
Default: clone from GitHub at `GIT_REF` (a version tag such as `v0.2.2`, or `main`). Alternative: `SOURCE = "dataset"` uses the highest-version copy under `/kaggle/input`.
Either way the code in `/kaggle/working/g1_rl` is replaced and `outputs/` is kept.

In [ ]:
import os, re, shutil, glob, subprocess
# ---- where to get the code -------------------------------------------------
SOURCE  = "github"       # "github" or "dataset"
GH_REPO = "plant7-arch/g1_rl"
GIT_REF = "v0.5.2"       # a version tag, or "main" for the latest push
# Private repo: add a Kaggle Secret named GITHUB_TOKEN (Add-ons -> Secrets), read-only token.
# ----------------------------------------------------------------------------
WORK = "/kaggle/working/g1_rl"

def pkg_version(root):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(f"{root}/g1_rl/__init__.py").read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)

if SOURCE == "github":
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None                       # public repo, or secret not attached
    auth = f"x-access-token:{token}@" if token else ""
    src = "/kaggle/tmp/g1_rl_src"
    shutil.rmtree(src, ignore_errors=True)
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", GIT_REF,
                        f"https://{auth}github.com/{GH_REPO}.git", src],
                       capture_output=True, text=True)
    if r.returncode:                        # never print the URL (it may contain the token)
        raise SystemExit("git clone failed: " + r.stderr.replace(token or "\0", "***")[-500:])
    sha = subprocess.run(["git", "-C", src, "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
    print(f"cloned {GH_REPO}@{GIT_REF} ({sha})")
else:
    roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
             if 'name = "g1_rl"' in open(p).read()]
    assert roots, "Attach the g1_rl dataset or use SOURCE='github'"
    roots.sort(key=pkg_version, reverse=True)
    for r in roots:
        print(f"found {'.'.join(map(str, pkg_version(r)))}  {r}")
    if len(roots) > 1 and pkg_version(roots[0]) == pkg_version(roots[1]):
        raise SystemExit("Two copies with the same version: delete the old zip from the dataset.")
    src = roots[0]

# Replace the code but keep outputs/ (checkpoints, logs, videos) from earlier sessions.
os.makedirs(WORK, exist_ok=True)
for item in os.listdir(WORK):
    if item != "outputs":
        p = os.path.join(WORK, item)
        shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
shutil.copytree(src, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".git"))
os.chdir(WORK)
print("g1_rl version", ".".join(map(str, pkg_version(WORK))), "|", sorted(os.listdir()))

## 2. Install GPU JAX + MuJoCo/MJX ⏱ ~2 min

In [ ]:
!bash scripts/install_kaggle.sh

---
## Phase 0: setup check & benchmark (done)

In [ ]:
# ⏱ ~2 min
# !python scripts/check_setup.py --platform gpu

In [ ]:
# 🌙 ~15 min
# !python scripts/benchmark.py --platform gpu --scaling --envs 1024,2048,4096,8192 --out outputs/benchmark.json

---
## Phase 1: environment
### Smoke test ⏱ ~3–4 min
512 robots with random actions, 60-step episodes and fast command changes. It prints PASS/FAIL checks and saves `outputs/phase1_smoke/` (report.json, trajectory, video).

In [ ]:
!python scripts/env_smoke.py --platform gpu

In [ ]:
from IPython.display import Video
Video("outputs/phase1_smoke/traj_env0.mp4", embed=True, width=480)

### Environment unit tests ⏱ ~4 min

In [ ]:
!pip install -q pytest && G1RL_TEST_PLATFORM=gpu python -m pytest -q tests/test_env.py

### Full test suite 🌙 ~8 min

In [ ]:
# !G1RL_TEST_PLATFORM=gpu python -m pytest -q

---
## Phase 2: rewards
### Reward unit tests ⏱ <1 min
Each term on hand-made inputs (no simulator).

In [ ]:
!pip install -q pytest && G1RL_TEST_PLATFORM=gpu python -m pytest -q tests/test_rewards.py

### Reward balance in the simulator ⏱ ~3 min
1024 robots with random actions. Prints every term's average weighted contribution per step and PASS/FAIL checks; saves `outputs/phase2_rewards.json`.

In [ ]:
!python scripts/reward_check.py --platform gpu

### Environment tests again ⏱ ~3 min (the env now computes the full reward)

In [ ]:
!G1RL_TEST_PLATFORM=gpu python -m pytest -q tests/test_env.py

---
## Phase 3: domain randomisation
### Randomiser unit tests ⏱ <1 min

In [ ]:
!G1RL_TEST_PLATFORM=gpu python -m pytest -q tests/test_randomize.py

### Randomisation in the simulator ⏱ ~4 min
Same rollout with the nominal robot (level 0) and full randomisation (level 1): parameter spread, push effect, falls, throughput. Saves `outputs/phase3_dr.json`.

In [ ]:
!python scripts/dr_check.py --platform gpu

### Environment tests again ⏱ ~3 min (latency, pushes and level handling are checked in the same step)

In [ ]:
!G1RL_TEST_PLATFORM=gpu python -m pytest -q tests/test_env.py

---
## Phase 4: PPO training
### PPO + checkpoint unit tests ⏱ ~1 min
GAE, Gaussian formulas, Adam, normaliser, device-local minibatches, a toy problem PPO must solve, checkpoint round trip and the exported-policy loader.

In [ ]:
!G1RL_TEST_PLATFORM=gpu python -m pytest -q tests/test_ppo.py tests/test_checkpoint.py

### Smoke training ⏱ ~4–5 min
1024 envs, 6 iterations of the real training step (compile dominates). Then the next cell resumes it.

In [ ]:
!rm -rf outputs/runs/smoke && python scripts/train.py --preset smoke --run-name smoke --stop-after 6

### Resume the smoke run ⏱ ~1–2 min (compiled code comes from the cache)

In [ ]:
!python scripts/train.py --preset smoke --run-name smoke --resume outputs/runs/smoke && ls outputs/runs/smoke

### Plot the smoke run ⏱ <1 min

In [ ]:
!python scripts/plot_metrics.py outputs/runs/smoke
from IPython.display import Image
Image("outputs/runs/smoke/training_curves.png")

**The overnight training run lives in `notebooks/kaggle_overnight.ipynb`** (separate notebook, so *Save & Run All* only runs the training).

---
## Next phases
Evaluation (5) and export (6) cells are added as each phase lands.